# Grupos por nodo y estados de carga unitarios (1 kgf en X, Y, Z)

Para cada nodo de piping:
1. Se asigna un **grupo** con el mismo nombre del nodo al joint de SAP que le corresponde.
2. Se crea un **estado de carga** (load pattern) con el mismo nombre del nodo.
3. Ese estado lleva una **carga de joint** de 1 kgf en X, Y y Z sobre ese joint.

**Entradas** (datasets de Kaggle):
- `.s2k` base exportado desde SAP2000, en unidades kgf-m.
- Excel/CSV con los nodos de piping: columna A = nombre del nodo. Columna B (opcional) = joint de SAP.

**Cómo se obtiene el joint de cada nodo** (en este orden):
1. Columna B del Excel/CSV, si está.
2. Un grupo que ya existe en el `.s2k` con el nombre del nodo (`10072` o `N 10072`).
3. Un joint cuya etiqueta es igual al nombre del nodo.

Los nodos sin joint se saltan y se listan. Un grupo que ya tenga otro joint, o con más de uno, detiene la ejecución.

**Salida:** `/kaggle/working/<modelo>_NODOS.s2k` y `/kaggle/working/mapeo_nodos.csv`.
Cada ejecución parte del `.s2k` base, así que puedes repetirla con un Excel modificado.

In [ ]:
import re
import glob
from pathlib import Path

import pandas as pd

INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

MODEL_FILE = None   # ruta fija al .s2k; None = detectar automáticamente
NODES_FILE = None   # ruta fija al Excel/CSV de nodos; None = detectar automáticamente

FORCE_KGF = (1.0, 1.0, 1.0)      # fuerza en kgf aplicada en X, Y, Z a cada estado
LOAD_CS = "GLOBAL"               # sistema de coordenadas de las cargas
ENCODING = "latin-1"             # preserva los bytes del .s2k al reescribirlo

In [ ]:
def find_single(exts):
    hits = sorted(p for p in glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
                  if Path(p).suffix.lower() in exts)
    if len(hits) != 1:
        raise FileNotFoundError(
            f"Se esperaba un archivo {sorted(exts)} en {INPUT_DIR}, encontrados: {hits}. "
            "Fija MODEL_FILE / NODES_FILE en la celda de configuración."
        )
    return hits[0]


model_path = MODEL_FILE or find_single({".s2k"})
if Path(model_path).stem.endswith("_NODOS"):
    raise ValueError("El .s2k de entrada es una salida de este notebook. Usa siempre el modelo base: cada ejecución parte de él.")
nodes_path = NODES_FILE or find_single({".xlsx", ".xls", ".csv"})
print("Modelo:", model_path)
print("Nodos:", nodes_path)

In [ ]:
def normalize_name(v):
    s = str(v).strip()
    return s[:-2] if re.fullmatch(r"\d+\.0", s) else s


def read_nodes(path):
    """Columna A: nombre del nodo. Columna B (opcional): joint de SAP. Una fila de cabecera 'nodo' se ignora."""
    if path.lower().endswith(".csv"):
        raw = pd.read_csv(path, header=None, dtype=str)
    else:
        raw = pd.read_excel(path, header=None, dtype=str)
    raw = raw.iloc[:, :2]
    if raw.shape[1] == 1:
        raw[1] = None
    raw.columns = ["node", "joint"]
    if str(raw.iloc[0]["node"]).strip().lower() in {"nodo", "nodos", "node"}:
        raw = raw.iloc[1:]

    df = raw.dropna(subset=["node"]).copy()
    df["node"] = df["node"].map(normalize_name)
    df["joint"] = df["joint"].map(lambda v: None if pd.isna(v) else normalize_name(v))

    dup = df[df.duplicated("node", keep=False)]
    if len(dup):
        raise ValueError(f"Nodos duplicados en el Excel:\n{dup.to_string()}")
    return df.reset_index(drop=True)


nodes_df = read_nodes(nodes_path)
print(f"{len(nodes_df)} nodos. Con joint explícito en el Excel: {nodes_df['joint'].notna().sum()}")
nodes_df.head()

In [ ]:
TABLE_RE = re.compile(r'^TABLE:\s+"(.+?)"')
KV_RE = re.compile(r'(\w+)=("[^"]*"|\S+)')


def parse_s2k(path):
    with open(path, encoding=ENCODING) as f:
        lines = f.read().splitlines()
    tables = {}
    current = None
    for line in lines:
        m = TABLE_RE.match(line.strip())
        if m:
            current = m.group(1)
            tables.setdefault(current, [])
        elif current is not None and "=" in line:
            tables[current].append({k: v.strip('"') for k, v in KV_RE.findall(line)})
    return lines, tables


def group_members(tables):
    """{grupo: {joints}} a partir de la tabla de asignaciones de grupos."""
    members = {}
    for name, rows in tables.items():
        if "GROUP" not in name.upper() or "ASSIGN" not in name.upper():
            continue
        for row in rows:
            if row.get("ObjectType", "").lower() == "joint":
                members.setdefault(row["GroupName"], set()).add(row["ObjectLabel"])
    return members


lines, tables = parse_s2k(model_path)

units = next((r["CurrUnits"] for r in tables.get("PROGRAM CONTROL", []) if "CurrUnits" in r), None)
if units is None:
    print("Aviso: no encontré CurrUnits en PROGRAM CONTROL; verifica a mano que el modelo esté en kgf-m.")
else:
    parts = [p.strip().lower() for p in units.split(",")]
    if parts[:2] != ["kgf", "m"]:
        raise ValueError(f"El modelo está en '{units}', no en kgf-m. Cambia unidades en SAP.")
    print("Unidades del modelo:", units)

members = group_members(tables)
joints = {r["Joint"] for r in tables.get("JOINT COORDINATES", []) if "Joint" in r}
defined_groups = {r["GroupName"] for r in tables.get("GROUPS 1 - DEFINITIONS", []) if "GroupName" in r}
defined_patterns = {r["LoadPat"] for r in tables.get("LOAD PATTERN DEFINITIONS", []) if "LoadPat" in r}
print(f"Joints: {len(joints)}. Grupos con joints: {len(members)}. Estados definidos: {len(defined_patterns)}")

In [ ]:
def resolve_joint(node, explicit):
    """Devuelve (joint, origen). joint es None si no hay forma de identificarlo."""
    if explicit is not None and not pd.isna(explicit):
        return explicit, "columna B del Excel"
    lookup = {g.lower(): g for g in members}
    for cand in (node, f"N {node}", f"N{node}"):
        if cand.lower() in lookup:
            js = sorted(members[lookup[cand.lower()]])
            if len(js) != 1:
                raise ValueError(f"nodo {node}: el grupo '{lookup[cand.lower()]}' tiene {len(js)} joints {js}; no se sabe a cuál aplicar la carga")
            return js[0], "grupo existente"
    if node in joints:
        return node, "etiqueta de joint"
    return None, None


mapping, origen, sin_joint = {}, {}, []
for r in nodes_df.itertuples():
    joint, how = resolve_joint(r.node, r.joint)
    if joint is None:
        sin_joint.append(r.node)
        continue
    if joints and joint not in joints:
        raise ValueError(f"nodo {r.node}: el joint {joint} no existe en JOINT COORDINATES")
    mapping[r.node] = joint
    origen[r.node] = how

mapeo = pd.DataFrame(
    [{"nodo": n, "joint": mapping[n], "origen": origen[n]} for n in mapping]
    + [{"nodo": n, "joint": None, "origen": "sin joint (se salta)"} for n in sin_joint]
)
print(f"Nodos con joint: {len(mapping)}. Sin joint (se saltan): {len(sin_joint)}")
if sin_joint:
    print("  " + ", ".join(sin_joint))
mapeo

In [ ]:
# Conflictos: un grupo con el nombre del nodo ya asignado a otro joint.
group_adds, def_adds, pattern_adds = [], [], []
for node, joint in mapping.items():
    if node in members:
        if members[node] != {joint}:
            raise ValueError(f"nodo {node}: el grupo '{node}' ya existe con joints {sorted(members[node])}, no con {joint}")
    else:
        group_adds.append((node, joint))
        if node not in defined_groups:
            def_adds.append(node)
    if node not in defined_patterns:
        pattern_adds.append(node)

print(f"Grupos nuevos: {len(group_adds)}. Definiciones de grupo nuevas: {len(def_adds)}. Estados nuevos: {len(pattern_adds)}")
print(f"Cargas de joint a escribir: {len(mapping)} (cada una {FORCE_KGF} kgf en X, Y, Z)")
if not mapping:
    raise ValueError("Ningún nodo tiene joint: revisa la columna B del Excel o los nombres de grupo/joint.")

In [ ]:
def fmt(v):
    return f'"{v}"'


def num(v):
    return f"{float(v):.10g}"


def add_rows(lines, table_name, new_rows):
    """Inserta filas justo debajo de la cabecera de la tabla; si no existe, la crea antes de END TABLE DATA."""
    if not new_rows:
        return lines
    for i, line in enumerate(lines):
        m = TABLE_RE.match(line.strip())
        if m and m.group(1) == table_name:
            return lines[:i + 1] + new_rows + lines[i + 1:]
    block = ["", f'TABLE:  "{table_name}"'] + new_rows
    end = next((i for i, l in enumerate(lines) if l.strip() == "END TABLE DATA"), len(lines))
    return lines[:end] + block + lines[end:]


def replace_existing(lines, pairs):
    """Quita las cargas de joint ya existentes para los pares (joint, estado) de este run."""
    keep, replaced, nonzero, table = [], 0, 0, None
    for line in lines:
        m = TABLE_RE.match(line.strip())
        if m:
            table = m.group(1)
        elif table == "JOINT LOADS - FORCE" and "=" in line:
            row = {k: v.strip('"') for k, v in KV_RE.findall(line)}
            if (row.get("Joint"), row.get("LoadPat")) in pairs:
                replaced += 1
                if any(float(row.get(k, 0)) != 0 for k in ["F1", "F2", "F3", "M1", "M2", "M3"]):
                    nonzero += 1
                continue
        keep.append(line)
    print(f"Cargas existentes reemplazadas: {replaced} (de ellas {nonzero} distintas de cero)")
    return keep


fx, fy, fz = (num(v) for v in FORCE_KGF)
group_def_rows = [
    f'   GroupName={fmt(g)}  Selection=No  SectionCut=No  Steel=No  Concrete=No  Aluminum=No  '
    f'ColdFormed=No  Stage=No  Bridge=No  AutoSeismic=No  AutoWind=No  SelDesSteel=No  '
    f'SelDesAlum=No  SelDesCold=No  MassWeight=No  Color=Blue'
    for g in def_adds
]
group_rows = [f"   GroupName={fmt(g)}  ObjectType=Joint  ObjectLabel={fmt(j)}" for g, j in group_adds]
pattern_rows = [f"   LoadPat={fmt(p)}  DesignType=Other  SelfWtMult=0" for p in pattern_adds]
joint_rows = [
    f"   Joint={fmt(mapping[n])}  LoadPat={fmt(n)}  CoordSys={LOAD_CS}  "
    f"F1={fx}  F2={fy}  F3={fz}  M1=0  M2=0  M3=0"
    for n in mapping
]

new_lines = replace_existing(lines, {(mapping[n], n) for n in mapping})
new_lines = add_rows(new_lines, "GROUPS 1 - DEFINITIONS", group_def_rows)
new_lines = add_rows(new_lines, "GROUPS 2 - ASSIGNMENTS", group_rows)
new_lines = add_rows(new_lines, "LOAD PATTERN DEFINITIONS", pattern_rows)
new_lines = add_rows(new_lines, "JOINT LOADS - FORCE", joint_rows)

out_path = Path(OUTPUT_DIR) / f"{Path(model_path).stem}_NODOS.s2k"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding=ENCODING, newline="\n") as f:
    f.write("\n".join(new_lines) + "\n")
mapeo.to_csv(Path(OUTPUT_DIR) / "mapeo_nodos.csv", index=False)
print(f"Escrito: {out_path}")
print(f"{len(joint_rows)} cargas de joint, {len(group_rows)} asignaciones de grupo, {len(pattern_rows)} estados nuevos")

In [ ]:
_, out_tables = parse_s2k(out_path)
out_members = group_members(out_tables)
out_patterns = {r["LoadPat"] for r in out_tables["LOAD PATTERN DEFINITIONS"]}
out_loads = {(r["Joint"], r["LoadPat"]): r for r in out_tables["JOINT LOADS - FORCE"]}

for node, joint in mapping.items():
    assert joint in out_members.get(node, set()), f"grupo {node} no contiene el joint {joint}"
    assert node in out_patterns, f"falta el estado {node}"
    row = out_loads.get((joint, node))
    assert row is not None, f"falta la carga en joint {joint}, estado {node}"
    got = [float(row[k]) for k in ("F1", "F2", "F3")]
    assert all(abs(g - e) < 1e-9 for g, e in zip(got, FORCE_KGF)), f"carga incorrecta en {node}: {got}"
print(f"Verificación OK: {len(mapping)} nodos con grupo, estado y carga de {FORCE_KGF} kgf")